## 🎯 과제 5 (선택): max_tokens로 답변 길이 제어 (실무 빈도 ★★★☆☆)

1교시에서 배운 `max_tokens` 파라미터를 적용해보세요. 토큰 폭주 방지와 비용 관리에 핵심입니다.

**요구사항**
- 모든 답변을 200 토큰 이내로 제한
- **힌트**: `ChatOpenAI(model="gpt-4o-mini", temperature=0.3, max_tokens=200)`

In [1]:
# 3️⃣ API 키 불러오기 (환경에 따라 자동 분기)
import os
import sys

# Colab에서 실행 중인지 자동 판별
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:

    from dotenv import load_dotenv
    load_dotenv()  # .env 파일에서 자동으로 불러옴

# 키가 잘 들어왔는지 확인 (앞 7글자만 출력 — 전체 노출은 위험)
api_key = os.environ.get("OPENAI_API_KEY", "")
print(f"API 키 로드: {api_key[:7]}..." if api_key else "❌ 키를 못 찾았어요")


API 키 로드: sk-proj...


In [2]:
# 1️⃣ OpenAI 클라이언트 생성 (한 번만 만들면 계속 재사용)
from openai import OpenAI

client = OpenAI()  # 환경변수의 OPENAI_API_KEY를 자동으로 읽어옴
print("✅ OpenAI 클라이언트 준비 완료")


✅ OpenAI 클라이언트 준비 완료


In [4]:
# 🔧 LLM 호출 헬퍼 함수 — 앞으로 계속 사용합니다
def ask_llm(messages, temperature=0.3, max_tokens=200):
    """메시지 리스트를 받아서 LLM의 답변 텍스트를 돌려준다."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content


In [20]:
# 페르소나 사전: 이름 → system 메시지
PERSONAS = {
    "분석가": {
        "system": "당신은 데이터 분석가입니다. 근거와 수치를 중심으로 논리적이고 간결하게 답변하세요.",
        "temperature": 0,
    },
    "디자이너": {
        "system": "당신은 UX/UI 디자이너입니다. 사용자 경험 관점에서 창의적이고 감각적으로 답변하세요.",
        "temperature": 0.8,
    },
}
print(f"✅ {len(PERSONAS)}개 페르소나 등록 완료: {list(PERSONAS.keys())}")

✅ 2개 페르소나 등록 완료: ['분석가', '디자이너']


In [18]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini")

prompt = ChatPromptTemplate.from_messages([
    ("system", "{role}"),
    ("human", "{question}"),
])

In [15]:
def ask(persona_name: str, question: str) -> str:
    # 1. 페르소나 설정 조회 (system 문구 + temperature)
    config = PERSONAS[persona_name]

    # 2. 페르소나별 temperature를 적용한 체인 (prompt | llm 패턴)
    persona_chain = prompt | llm.bind(temperature=config["temperature"])

    # 3. 체인 호출 후 답변 텍스트만 반환
    result = persona_chain.invoke({"role": config["system"], "question": question})
    return result.content

In [16]:
# 명령어 처리 함수: (새로운_페르소나, 종료여부) 튜플 반환
def handle_command(user_input: str, current_persona: str):
    parts = user_input.split(maxsplit=1)   # "/change 마케터" → ["/change", "마케터"]
    cmd = parts[0]

    if cmd in ("/quit", "/exit"):
        print("👋 종료합니다.")
        return current_persona, True       # 종료 플래그 True

    if cmd == "/persona":
        print(f"🎭 현재 페르소나: {current_persona}")

    elif cmd == "/help":
        print("📖 명령어: /quit, /exit, /change <이름>, /persona, /help")

    elif cmd == "/change":
        new_name = parts[1] if len(parts) > 1 else ""
        if new_name in PERSONAS:
            print(f"✅ 페르소나가 '{new_name}'(으)로 변경되었습니다.")
            return new_name, False         # 페르소나만 변경
        else:
            print(f"❌ '{new_name}'은 등록된 페르소나가 아닙니다.")
            print(f"   사용 가능: {', '.join(PERSONAS.keys())}")

    else:
        print(f"❓ 알 수 없는 명령어: {cmd} (/help 입력)")

    return current_persona, False

In [21]:
# 챗봇 실행 함수
def run_chatbot(initial_persona: str = "분석가"):
    persona = initial_persona
    print(f"👋 페르소나 챗봇입니다. (현재: {persona})")
    print("   /help 로 명령어 보기, /quit 로 종료\n")

    while True:
        user_input = input("질문> ").strip()
        if not user_input:                       # 빈 입력은 무시
            continue

        if user_input.startswith("/"):           # 명령어 처리
            persona, should_quit = handle_command(user_input, persona)
            if should_quit:
                break
        else:                                    # 일반 질문 처리
            answer = ask(persona, user_input)
            print(f"[{persona}] {answer}\n")

run_chatbot("분석가")

👋 페르소나 챗봇입니다. (현재: 분석가)
   /help 로 명령어 보기, /quit 로 종료

[분석가] 안녕하세요! 데이터 분석이나 관련 질문이 있으시면 말씀해 주세요. 도움이 필요하신 부분에 대해 논리적이고 간결하게 답변해 드리겠습니다.

[분석가] 네, F1 Score 외에도 다양한 알고리즘 모델 선택 기준이 있습니다. 주요 기준은 다음과 같습니다:

1. **정확도 (Accuracy)**: 전체 예측 중에서 맞춘 비율로, 클래스 불균형이 심하지 않은 경우 유용합니다.

2. **정밀도 (Precision)**: 양성으로 예측한 것 중 실제 양성의 비율로, False Positive를 줄이는 것이 중요한 경우에 사용됩니다.

3. **재현율 (Recall)**: 실제 양성 중에서 양성으로 예측한 비율로, False Negative를 줄이는 것이 중요한 경우에 적합합니다.

4. **ROC-AUC (Receiver Operating Characteristic - Area Under Curve)**: 다양한 임계값에서의 True Positive Rate와 False Positive Rate를 비교하여 모델의 성능을 평가합니다. 이 값이 1에 가까울수록 좋습니다.

5. **로그 손실 (Log Loss)**: 예측 확률과 실제 레이블 간의 차이를 측정하여, 확률적 예측의 품질을 평가합니다.

6. **Mean Absolute Error (MAE) / Mean Squared Error (MSE)**: 회귀 문제에서 예측값과 실제값 간의 차이를 측정하는 지표입니다. MAE는 절대 오차의 평균, MSE는 제곱 오차의 평균입니다.

7. **모델의 복잡도**: 모델의 해석 가능성, 학습 시간, 예측 시간 등을 고려하여 선택할 수 있습니다. 간단한 모델이 더 나은 해석력을 제공할 수 있습니다.

8. **교차 검증 (Cross-Validation)**: 모델의 일반화 성능을 평가하기 위해 데이터를 여러 번 나누어 학습하고 검증하는 방법입니다.

이러한 기준들은 문제